# 05b · Verificación en notebook limpio
Carga el champion elegido en el PSet #3 (versión de la corrida final `rol=champion`) y comprueba que reproduce **exactamente** la métrica de prueba registrada en MLflow. Ejecutar en un kernel nuevo (*Restart* → *Run All*), después del `05`.

In [1]:
import mlflow, mlflow.sklearn, numpy as np
from sklearn.metrics import average_precision_score
from src.datos import cargar_obt, cargar_obt_entrenamiento
from src.pipeline import *
from src.experimento import NOMBRE_MODELO, configurar
configurar()
cli = mlflow.MlflowClient()
# Champion elegido en el PSet #3 (corrida final con rol=champion). En el proyecto final el alias
# champion puede apuntar a una versión reentrenada; esta verificación es sobre la versión original.
versiones = cli.search_model_versions(f"name='{NOMBRE_MODELO}'")
mv = next(v for v in sorted(versiones, key=lambda v: int(v.version))
          if cli.get_run(v.run_id).data.tags.get("fase") == "final" and cli.get_run(v.run_id).data.tags.get("rol") == "champion")
run = cli.get_run(mv.run_id)
modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}/{mv.version}")
print("Alias champion actual → versión", cli.get_model_version_by_alias(NOMBRE_MODELO, "champion").version)
print("Modelo:", NOMBRE_MODELO, "| versión", mv.version, "| run_id", mv.run_id)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Alias champion actual → versión 4
Modelo: resenas-bancos-ec-modelo | versión 1 | run_id 972c63a668334b73a195ce78b0bb44ea


In [2]:
df = agregar_features(cargar_obt_entrenamiento())
train, test = split_temporal(df, 0.2)
p = modelo.predict_proba(test[FEATURES])[:, 1]
umbral = run.data.metrics["umbral_aplicado"]
pr_auc = average_precision_score(test[TARGET], p)
costo = costo_negocio(test[TARGET], (p >= umbral).astype(int))
print(f"PR-AUC prueba  → reproducida: {pr_auc:.6f} | registrada: {run.data.metrics['test_pr_auc']:.6f}")
print(f"Costo negocio  → reproducido: {costo:.6f} | registrado: {run.data.metrics['test_costo']:.6f}")
# Tolerancia: los embeddings se recalculan en otra compilación de torch/CPU (diferencias ~1e-7 en la
# probabilidad). 1e-4 en PR-AUC es 100 veces menor que la desviación entre pliegues (0,0145); en costo,
# menos de una decisión distinta en 4 340 reseñas (5/4340 = 0,0012).
dif_pr, dif_costo = abs(pr_auc - run.data.metrics["test_pr_auc"]), abs(costo - run.data.metrics["test_costo"])
print(f"Diferencias: PR-AUC {dif_pr:.2e} | costo {dif_costo:.2e}")
assert dif_pr < 1e-4 and dif_costo < 5 / len(test)
print("OK: el modelo reproduce las métricas registradas dentro de la tolerancia numérica.")

/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


PR-AUC prueba  → reproducida: 0.890997 | registrada: 0.890997
Costo negocio  → reproducido: 0.170046 | registrado: 0.170046
Diferencias: PR-AUC 1.48e-07 | costo 0.00e+00
OK: el modelo reproduce las métricas registradas dentro de la tolerancia numérica.
